In [ ]:
pip install requests

In [ ]:
pip install flask flask-cors

In [ ]:
pip install langchain langchain_community langchain-ollama

In [21]:
# --- Step 1: Load Data from SQLite and Create Documents ---
import sqlite3
from langchain.schema import Document

sqlite_file = "./courses.sqlite"
table_name = "merged_courses"
content_columns = [
    "title", "instructor", "learning_obj", "course_contents", "prerequisites",
    "readings", "applicability", "workload", "credits", "evaluation", 
    "time", "duration", "course_type"
]

def load_db(sqlite_file, table_name, content_columns):
    conn = sqlite3.connect(sqlite_file)
    cursor = conn.cursor()
    
    # Query all rows
    query = f"SELECT * FROM {table_name}"
    cursor.execute(query)
    
    rows = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]
    conn.close()

    # Combine content columns into `page_content` and include metadata
    documents = [
        Document(
            page_content=" ".join(str(row[column_names.index(col)]) for col in content_columns),
            metadata={col: row[column_names.index(col)] for col in column_names}
        )
        for row in rows
    ]
    return documents

documents = load_db(sqlite_file, table_name, content_columns)
print(documents[:2])

[Document(metadata={'rowid': 1, 'file_loc': 'E:\\HSM\\project_hmi\\webappcoursespot\\courses\\zqm_modul\\en\\Automotive Technology Mgmt_MD.pdf', 'title': 'Automotive Technology Management', 'instructor': 'Prof Dr Michael Dornieden', 'learning_obj': 'Identify relevant market framework conditions of carmakers\r▪ Realize the economic importance of the automotive industry\r▪ Know automotive key figures in respect to particular countries\r▪ Categorize the product portfolio of automotive suppliers\r▪ Examine and understand future trends of automotive markets and\rresource requirements of carmakers (e.g. electromobility)\r▪ Critically evaluate strategic alliances between carmakers currently in\rpractical company use and develop an understanding of how they can\rbe improved in order to reach the desired organizational goals\r▪ Know theoretical foundations of product lifecycle management\r▪ Characterize the six phases of the Generic Product Development\rProcess and demonstrate its application t

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=150, add_start_index=True
)
all_splits = text_splitter.split_documents(documents)

In [6]:
pip install langchain-community

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Step 3: Persist the vector store
import os
from langchain_community.embeddings import OllamaEmbeddings
# from langchain_chroma import Chroma
from langchain_community.vectorstores.chroma import Chroma
import logging

# Set the path for the vector store
VECTORSTORE_DIR = "./chroma_vectorstore"
def initialize_vectorstore():
    if not os.path.exists(VECTORSTORE_DIR):
        # If the directory doesn't exist, create and persist the vector store
        local_embeddings = OllamaEmbeddings(model="all-minilm")
        vectorstore = Chroma.from_documents(
            documents=all_splits,
            embedding=local_embeddings,
            persist_directory=VECTORSTORE_DIR
        )
        vectorstore.persist()
        logging.info("Vector store created and persisted to disk.")
    else:
        logging.info("Vector store already exists. Skipping embedding computation.")

# Initialize the vector store (only computes embeddings if not already persisted)
initialize_vectorstore()

/var/folders/g2/g3tj8_9x6cv7gfxf821v0scr0000gn/T/ipykernel_2117/1522872885.py:13: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  local_embeddings = OllamaEmbeddings(model="all-minilm")
/var/folders/g2/g3tj8_9x6cv7gfxf821v0scr0000gn/T/ipykernel_2117/1522872885.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [24]:
# Load the vector store for querying
def load_vectorstore():
    local_embeddings = OllamaEmbeddings(model="all-minilm")
    return Chroma(
        persist_directory=VECTORSTORE_DIR,
        embedding_function=local_embeddings
    )

vectorstore = load_vectorstore()

# print(vectorstore)

In [10]:
pip install langchain-chroma

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [29]:
# from langchain_community.embeddings import OllamaEmbeddings
# from langchain_chroma import Chroma
from langchain_ollama.llms import OllamaLLM

# local_embeddings = OllamaEmbeddings(model="all-minilm")
# vectorstore = Chroma.from_documents(documents=all_splits, embedding=local_embeddings)

# llm = OllamaLLM(model="llama3.1")

In [30]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [13]:
pip install RapidFuzz

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.4 MB 1.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [32]:
from rapidfuzz import process

# Step 4: Define a function for query correction using fuzzy matching
def correct_query(query, documents, top_n=1):
    # Extract titles or key text from documents for fuzzy matching
    document_titles = [doc.metadata.get("title", "") for doc in documents]
    
    # Use fuzzy matching to find the closest matches
    best_matches = process.extract(query, document_titles, limit=top_n)
    corrected_query = best_matches[0][0] if best_matches else query  # Take the best match
    return corrected_query

In [33]:
llm = OllamaLLM(model="llama3.1")

In [34]:




# Define the RAG application class
class RAGApplication:
    def __init__(self, retriever):
        self.retriever = retriever
        # self.rag_chain = rag_chain
    def run(self, question):
        corrected_query = correct_query(question, documents)
        logging.info(f"Original Query: {question} | Corrected Query: {corrected_query}")

        # retrieved_docs = retriever.invoke(question)
        # context = ' '.join([doc.page_content for doc in retrieved_docs])
        retrieved_docs = self.retriever.get_relevant_documents(corrected_query)
        context = ' '.join([doc.page_content for doc in retrieved_docs])
        
        # Get the answer from the language model
        response = llm.invoke(f"""Answer the question according to the context given very briefly:
                Question: {question}.
                Context: {context}
                """)
        
        print(response, 'response')
        return {
            "response": response,
            "context": context
        }

In [18]:
pip install --upgrade flask-cors


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [35]:
from flask import Flask, request, jsonify
from flask_cors import CORS, cross_origin


# Initialize Flask app
app = Flask(__name__)
CORS(app, support_credentials=True)

# Initialize logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

@app.route("/ask", methods=["POST"])
@cross_origin(supports_credentials=True)
def ask():
    """
    Endpoint to handle user questions and return responses from the RAG application.
    """
    try:
        # Parse JSON data from request
        data = request.json
        if not data or "query" not in data:
            logging.warning("No query provided in the request.")
            return jsonify({"error": "No query provided"}), 400

        question = data["query"].strip()
        if not question:
            logging.warning("Empty query provided in the request.")
            return jsonify({"error": "Query cannot be empty"}), 400

        # Process the question using RAGApplication
        rag_application = RAGApplication(retriever)
        answer = rag_application.run(question)

        logging.info(f"Question: {question} | Answer: {answer}")
        return jsonify({
            "response": answer["response"],
            "context": answer["context"]
        })

    except Exception as e:
        logging.error(f"Error occurred: {str(e)}", exc_info=True)
        return jsonify({'error': "An internal error occurred. Please try again later."}), 500

# Run the app
if __name__ == "__main__":
    app.run(debug=True, host='0.0.0.0', port=5050, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5050
 * Running on http://192.168.0.5:5050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Dec/2024 21:47:20] "POST /ask HTTP/1.1" 200 -


The subject of "Mobile Systems (Mobile Systeme)" involves the development of smart, mobile applications that incorporate concepts such as location-based services, communication in mobile apps, acquisition of environmental data using sensor interfaces, activity recognition, and track & trace-applications. The focus is on programming with Java and Android. response


INFO:werkzeug:127.0.0.1 - - [01/Dec/2024 21:49:05] "POST /ask HTTP/1.1" 200 -


The instructor for "Mobile Systems" is Prof. Dr. Michael Cebulla. response


In [ ]:
# question = "what is Computational Intelligence?"
# question = "What is the difference between IT Security and Cheese Cake?"
# retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
# retrieved_docs = retriever.invoke(question)

In [ ]:
# context = ' '.join([doc.page_content for doc in retrieved_docs])

In [ ]:
pip install -U langchain-ollama

In [ ]:
# from langchain_ollama.llms import OllamaLLM

# llm = OllamaLLM(model="llama3.1")
# response = llm.invoke(f"""Answer the question according to the context given very briefly:
#            Question: {question}.
#            Context: {context}
# """)

In [ ]:
# print(response)